[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/07_soil-data/07_02_soil_data_gnatsgo.ipynb)


# Module 7, Lesson 2: Gridded Soil Data with gNATSGO
## Cloud raster and table workflows for watershed soil screening

### Lesson Level
Beginner to Intermediate

### Purpose
Lesson 1 used SSURGO polygons and the Soil Data Access service. This lesson answers similar point and watershed questions with the gridded National Soil Survey Geographic Database, called gNATSGO. You will discover a cloud raster tile, sample soil data at a point, clip two rasters to a watershed, join raster map-unit keys to a soil attribute table, summarize hydrologic soil groups by area, and export traceable results.

### What You'll Work Through Today
- Understand how gNATSGO relates to SSURGO, STATSGO2, and Raster Soil Survey data
- Search the Microsoft Planetary Computer catalog for the correct gNATSGO tile
- Sample map-unit key and available water storage at one point
- Clip cloud-optimized rasters to a HUC-12 watershed
- Join raster cells to the gNATSGO `muaggatt` table by `mukey`
- Calculate raster coverage, map-unit area, HSG area, and watershed AWS statistics
- Export summary tables and clipped GeoTIFFs

### Learning Objectives
By the end of this lesson, you will be able to:
- Explain the difference between SSURGO polygons and a gNATSGO map-unit grid
- Use a STAC catalog to find spatial data without hard-coding a file URL
- Inspect raster CRS, resolution, NoData, and grid alignment
- Calculate area from raster metadata
- Join a categorical raster to a tabular soil attribute using a stable key
- Interpret AWS and hydrologic soil group without confusing them with Ksat

### Prerequisites
- Module 1 for Python and Colab basics
- Module 3 for watershed vectors and coordinate reference systems
- Module 4 for raster clipping and NoData concepts
- Module 7, Lesson 1 for SSURGO map units, components, horizons, HSG, and Ksat


## Using AI in This Lesson

AI can help explain STAC objects, raster metadata, and table joins. It cannot decide whether a soil value is appropriate for a hydrologic model. Check every unit, source date, aggregation method, and engineering assumption yourself.

🤖 **Try asking your AI assistant:** *"Explain the difference between a soil map-unit polygon and a raster whose cell value is a map-unit key. Why does the raster still need an attribute table?"*


## Part 1: Mental Model - A Tiled Soil Index Map 🗂️

Think of gNATSGO as a national soil index map printed on a regular grid. Each grid cell stores a `mukey`, which is the map-unit key you met in Lesson 1. The cell does not hold every soil property. It points to a map-unit record in a related table.

The workflow has three pieces:

1. **Catalog:** find the cloud tile that covers the watershed.
2. **Raster:** read map-unit keys or a prepared soil attribute from grid cells.
3. **Table:** join each map-unit key to names, hydrologic soil group, drainage class, and other attributes.

| Lesson 1: SSURGO | Lesson 2: gNATSGO |
|---|---|
| Map units are polygons | Map units are represented by grid-cell values |
| SDA finds records with SQL and geometry | STAC finds a tile and Parquet supplies attributes |
| Spatial area comes from clipped polygons | Spatial area comes from counted cells |
| Detailed SSURGO where mapped | Composite coverage using SSURGO, STATSGO2, and RSS |

A 10 meter cell is a storage format, not a claim that every soil boundary is known to 10 meter accuracy. The source survey scale and source type still control how much confidence the data deserve.


## Part 2: Data Source, Citation, and Engineering Scenario 🌐

The USDA Natural Resources Conservation Service describes gNATSGO as a composite database that combines SSURGO, STATSGO2, and Raster Soil Survey data into seamless coverage for the United States and its territories. NRCS refreshes published soil databases annually. NRCS also notes that gNATSGO has no vectorized map-unit layer, its raster joins to tables on `mukey`, and grid resolution depends on the delivered product.

This lesson uses the Microsoft Planetary Computer distribution:

- [NRCS gNATSGO description and current citation](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo)
- [Planetary Computer gNATSGO rasters](https://planetarycomputer.microsoft.com/dataset/gnatsgo-rasters)
- [Planetary Computer gNATSGO collection overview](https://planetarycomputer.microsoft.com/dataset/group/gnatsgo)
- [Planetary Computer STAC documentation](https://planetarycomputer.microsoft.com/docs/quickstarts/reading-stac/)
- [Planetary Computer tabular-data documentation](https://planetarycomputer.microsoft.com/docs/quickstarts/reading-tabular-data/)

**NRCS citation shown on the source page at the time this lesson was prepared:**

> Soil Survey Staff (2026). Gridded national soil survey geographic (gNATSGO) database. USDA Natural Resources Conservation Service. https://nrcs.app.box.com/v/soils

The Planetary Computer collection used here is an archived snapshot dated July 2020. It is useful for a reproducible cloud-data lesson, but it is not the current annual NRCS release. Check the NRCS page when a project requires the latest soil database.

### Today's Engineering Scenario
You are screening the same Town of South Greeley HUC-12 used in Lesson 1. You need a grid-based view of hydrologic soil group and available water storage for watershed comparison and early model setup. You also need enough QA/QC to show that the grid covers the watershed and that the raster and table units agree.


## Part 3: Workspace Setup 🛠️

This cell installs only packages that are missing. In Colab, the Planetary Computer packages normally need to be added once per session.


In [ ]:
import importlib.util
import subprocess
import sys

package_map = {
    'pystac_client': 'pystac-client',
    'planetary_computer': 'planetary-computer',
    'adlfs': 'adlfs',
    'pyarrow': 'pyarrow',
}

missing_packages = [
    package_name
    for import_name, package_name in package_map.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print(f"Installing: {', '.join(missing_packages)}")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', *missing_packages
    ])
else:
    print('Required cloud-data packages are already installed.')


In [ ]:
# Tabular and numerical data
import datetime
import numpy as np
import pandas as pd

# Vector and raster data
import geopandas as gpd
import rasterio
import rasterio.mask
from rasterio.plot import plotting_extent

# Cloud catalog and signed data access
import planetary_computer
import pystac_client

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

print('Libraries imported. Ready to work with gNATSGO.')


### Uploading the Watershed Boundary

Upload the same watershed ZIP used in Lesson 1. The gNATSGO rasters and table are read from the cloud, so they do not need to be uploaded.


In [ ]:
from google.colab import files

print('Please upload this file:')
print('NHD__Watershed_Boundaries_HUC_12_Selected.zip')
uploaded = files.upload()

print(f"Uploaded {len(uploaded)} file(s)")


In [ ]:
watersheds = gpd.read_file(
    'zip://NHD__Watershed_Boundaries_HUC_12_Selected.zip'
)

target_huc = '101900090108'
target_watershed = watersheds[
    watersheds['HUC12'].astype(str) == target_huc
].copy()

if target_watershed.empty:
    raise RuntimeError(f'HUC-12 {target_huc} was not found in the uploaded file.')

target_wgs84 = target_watershed.to_crs(4326)

print(f"Loaded {len(watersheds)} watershed polygons")
print(f"Target: {target_watershed['Name'].iloc[0]}")
print(f"HUC-12: {target_huc}")
print(f"Reported area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")
print(f"Watershed bounds in WGS84: {target_wgs84.total_bounds}")


## Part 4: Find the gNATSGO Tile with STAC 🔎

STAC stands for SpatioTemporal Asset Catalog. Think of it as a searchable card catalog for cloud geospatial files. We ask which gNATSGO raster item intersects the watershed. The catalog returns metadata and signed links to cloud-optimized GeoTIFFs.

A signed link contains a temporary access token. Use it during the current session, but do not save it in a report, CSV, notebook output, or source-control commit.


In [ ]:
STAC_API = 'https://planetarycomputer.microsoft.com/api/stac/v1'

try:
    catalog = pystac_client.Client.open(
        STAC_API,
        modifier=planetary_computer.sign_inplace,
    )
except Exception as error:
    raise RuntimeError(
        'The Planetary Computer catalog could not be opened. Check your '
        'internet connection and run this cell again.'
    ) from error

raster_collection = catalog.get_collection('gnatsgo-rasters')
table_collection = catalog.get_collection('gnatsgo-tables')

print(raster_collection.title)
print(table_collection.title)


In [ ]:
search = catalog.search(
    collections=['gnatsgo-rasters'],
    intersects=target_wgs84.geometry.iloc[0].__geo_interface__,
)
gnatsgo_items = list(search.items())

if len(gnatsgo_items) == 0:
    raise RuntimeError('No gNATSGO raster tile intersects this watershed.')
if len(gnatsgo_items) > 1:
    raise RuntimeError(
        'This watershed crosses more than one gNATSGO tile. Mosaicking is '
        'outside this beginner lesson. Choose another course watershed or '
        'extend the workflow with rasterio.merge.'
    )

gnatsgo_item = gnatsgo_items[0]
dataset_snapshot_date = gnatsgo_item.datetime.date().isoformat()

print(f"Matching item: {gnatsgo_item.id}")
print(f"Dataset snapshot date: {dataset_snapshot_date}")
print(f"Item bounds: {gnatsgo_item.bbox}")
print(f"Available assets: {len(gnatsgo_item.assets)}")


In [ ]:
selected_assets = ['mukey', 'aws0_100']
asset_rows = []

for asset_name in selected_assets:
    asset_definition = raster_collection.extra_fields['item_assets'][asset_name]
    asset_rows.append({
        'asset': asset_name,
        'title': asset_definition['title'],
        'description': asset_definition['description'].split('\n')[0],
        'unsigned_file': gnatsgo_item.assets[asset_name].href.split('?')[0],
    })

asset_catalog = pd.DataFrame(asset_rows)
asset_catalog[['asset', 'title', 'description']]


### Why These Two Rasters

- `mukey` is categorical. Its cell value identifies a soil map unit and joins to related tables. Never average map-unit keys.
- `aws0_100` is continuous. It stores the map-unit weighted estimate of available water storage from 0 to 100 cm, expressed in millimeters. It can be summarized with area-weighted statistics.

Available water storage is the volume of water a soil can store that is available to plants. It is not saturated hydraulic conductivity, infiltration capacity, initial loss, or a calibrated model storage parameter.


## Part 5: Point Query with Raster Sampling 📍

Lesson 1 asked SDA which map unit contains a point. Here, the point is transformed to the raster CRS and sampled directly from the cloud raster. We use a representative point guaranteed to fall inside the watershed.


In [ ]:
representative_point = target_wgs84.geometry.iloc[0].representative_point()
point_lon = representative_point.x
point_lat = representative_point.y
point_gdf = gpd.GeoDataFrame(
    {'name': ['representative point']},
    geometry=[representative_point],
    crs=4326,
)

print(f"Point longitude: {point_lon:.6f}")
print(f"Point latitude:  {point_lat:.6f}")


In [ ]:
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open(gnatsgo_item.assets['mukey'].href) as src:
        point_in_raster_crs = point_gdf.to_crs(src.crs)
        point_xy = [(
            point_in_raster_crs.geometry.iloc[0].x,
            point_in_raster_crs.geometry.iloc[0].y,
        )]
        point_mukey_value = next(src.sample(point_xy))[0]
        point_mukey_nodata = src.nodata

    with rasterio.open(gnatsgo_item.assets['aws0_100'].href) as src:
        point_aws_mm = float(next(src.sample(point_xy))[0])
        point_aws_nodata = src.nodata

if point_mukey_value == point_mukey_nodata:
    raise RuntimeError('The representative point returned NoData for mukey.')
if point_aws_mm == point_aws_nodata:
    raise RuntimeError('The representative point returned NoData for AWS.')

point_mukey = int(point_mukey_value)

print(f"Map-unit key at point: {point_mukey}")
print(f"AWS 0 to 100 cm at point: {point_aws_mm:.1f} mm")


The point tells us which grid cell and map unit were sampled. We will add the map-unit name, drainage class, and HSG after reading the watershed's table rows once. This avoids making the same cloud table request twice.

🤖 **Try asking your AI assistant:** *"Why is averaging a mukey raster meaningless, while averaging an available-water-storage raster can be valid? Include units in your answer."*


## Part 6: Watershed Raster Workflow 🗺️

Cloud-optimized GeoTIFFs support range requests. Rasterio can read only the blocks needed around our watershed instead of downloading the full 16,384 by 16,384 tile. The helper below clips one asset and returns its data and metadata.


In [ ]:
def clip_gnatsgo_asset(asset_name):
    """Clip one signed gNATSGO raster asset to the target watershed."""
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(gnatsgo_item.assets[asset_name].href) as src:
                watershed_in_raster_crs = target_watershed.to_crs(src.crs)
                geometry = [
                    watershed_in_raster_crs.geometry.iloc[0].__geo_interface__
                ]
                clipped, clipped_transform = rasterio.mask.mask(
                    src, geometry, crop=True
                )
                profile = src.profile.copy()
                metadata = {
                    'crs': src.crs,
                    'resolution': src.res,
                    'nodata': src.nodata,
                    'dtype': src.dtypes[0],
                }
    except Exception as error:
        raise RuntimeError(
            f"The {asset_name} cloud raster could not be read. The signed "
            'link may have expired, or the service may be unavailable. Run '
            'the catalog and search cells again, then retry.'
        ) from error

    return clipped[0], clipped_transform, profile, metadata


print('clip_gnatsgo_asset() is ready.')


In [ ]:
mukey_array, mukey_transform, mukey_profile, mukey_metadata = (
    clip_gnatsgo_asset('mukey')
)
aws_array, aws_transform, aws_profile, aws_metadata = (
    clip_gnatsgo_asset('aws0_100')
)

print('mukey metadata:', mukey_metadata)
print('AWS metadata:', aws_metadata)
print(f"Clipped array shape: {mukey_array.shape}")


In [ ]:
same_grid = (
    mukey_array.shape == aws_array.shape
    and mukey_transform.almost_equals(aws_transform)
    and mukey_metadata['crs'] == aws_metadata['crs']
    and mukey_metadata['resolution'] == aws_metadata['resolution']
)

if not same_grid:
    raise RuntimeError(
        'The mukey and AWS rasters are not aligned. Do not combine their '
        'cells until CRS, resolution, shape, and transform all match.'
    )

cell_area_m2 = abs(
    mukey_transform.a * mukey_transform.e
    - mukey_transform.b * mukey_transform.d
)
cell_area_acres = cell_area_m2 / 4046.86

print('Grid alignment QA/QC passed.')
print(f"Cell size: {mukey_metadata['resolution']} meters")
print(f"Cell area from transform: {cell_area_m2:.0f} square meters")


### Coverage QA/QC

A complete-looking map can still hide NoData gaps. Compare valid grid area with polygon area before trusting the summary. Small differences are expected because a raster approximates a curved boundary with square cells.


In [ ]:
mukey_nodata = mukey_metadata['nodata']
aws_nodata = aws_metadata['nodata']

valid_mukey_mask = mukey_array != mukey_nodata
valid_aws_mask = (
    (aws_array != aws_nodata)
    & np.isfinite(aws_array)
)

watershed_raster_crs = target_watershed.to_crs(mukey_metadata['crs'])
polygon_area_m2 = watershed_raster_crs.geometry.area.sum()
mukey_valid_area_m2 = valid_mukey_mask.sum() * cell_area_m2
aws_valid_area_m2 = valid_aws_mask.sum() * cell_area_m2

mukey_coverage_pct = 100 * mukey_valid_area_m2 / polygon_area_m2
aws_coverage_pct = 100 * aws_valid_area_m2 / polygon_area_m2

print(f"Polygon area: {polygon_area_m2 / 4046.86:,.1f} acres")
print(f"Valid mukey grid area: {mukey_valid_area_m2 / 4046.86:,.1f} acres")
print(f"Valid AWS grid area: {aws_valid_area_m2 / 4046.86:,.1f} acres")
print(f"mukey coverage: {mukey_coverage_pct:.2f}%")
print(f"AWS coverage: {aws_coverage_pct:.2f}%")

if mukey_coverage_pct < 98 or aws_coverage_pct < 98:
    print('WARNING: Coverage is below 98%. Investigate before using results.')
else:
    print('Coverage QA/QC passed for both rasters.')


### Count Map-Unit Cells

The `mukey` raster is categorical. Count cells in each map unit, multiply by cell area, and divide by total valid cells for a watershed percentage.


In [ ]:
mukey_values, mukey_cell_counts = np.unique(
    mukey_array[valid_mukey_mask], return_counts=True
)

mapunit_area = pd.DataFrame({
    'mukey': mukey_values.astype('int64'),
    'cell_count': mukey_cell_counts.astype('int64'),
})
mapunit_area['area_acres'] = mapunit_area['cell_count'] * cell_area_acres
mapunit_area['percent_valid_grid'] = (
    100 * mapunit_area['cell_count'] / mapunit_area['cell_count'].sum()
)
mapunit_area = mapunit_area.sort_values(
    'area_acres', ascending=False
).reset_index(drop=True)

print(f"Found {len(mapunit_area)} map units in the watershed grid")
print(mapunit_area.round(2).to_string(index=False))


### Join Map Units to the gNATSGO Table

The Planetary Computer stores gNATSGO tables in Parquet format. Parquet lets us request only the columns and map-unit rows we need. The first read can take about 20 to 40 seconds because the service must inspect a large national table.


In [ ]:
def read_gnatsgo_table(table_name, columns, filters=None):
    """Read selected columns and rows from one gNATSGO Parquet table."""
    try:
        table_item = table_collection.get_item(table_name)
        if table_item is None:
            raise ValueError(f'Table item not found: {table_name}')
        table_asset = table_item.assets['data']
        return pd.read_parquet(
            table_asset.href,
            storage_options=table_asset.extra_fields['table:storage_options'],
            columns=columns,
            filters=filters,
        )
    except Exception as error:
        raise RuntimeError(
            f"The gNATSGO table '{table_name}' could not be read. Check the "
            'internet connection, rerun the catalog cell to refresh signed '
            'access, and try again.'
        ) from error


print('read_gnatsgo_table() is ready.')


In [ ]:
watershed_mukeys = [int(value) for value in mapunit_area['mukey']]

muaggatt = read_gnatsgo_table(
    'muaggatt',
    columns=[
        'mukey', 'musym', 'muname', 'hydgrpdcd', 'drclassdcd',
    ],
    filters=[('mukey', 'in', watershed_mukeys)],
)

valu1 = read_gnatsgo_table(
    'valu1',
    columns=['mukey', 'aws0_100'],
    filters=[('mukey', 'in', watershed_mukeys)],
)

muaggatt_mukeys = set(muaggatt['mukey'].astype(int))
valu1_mukeys = set(valu1['mukey'].astype(int))
missing_muaggatt_mukeys = sorted(set(watershed_mukeys) - muaggatt_mukeys)
missing_valu1_mukeys = sorted(set(watershed_mukeys) - valu1_mukeys)

print(f"Requested {len(watershed_mukeys)} map units")
print(f"Received {len(muaggatt)} muaggatt rows")
print(f"Received {len(valu1)} valu1 rows")
if missing_muaggatt_mukeys or missing_valu1_mukeys:
    print(f"WARNING: Missing muaggatt keys: {missing_muaggatt_mukeys}")
    print(f"WARNING: Missing valu1 keys: {missing_valu1_mukeys}")
else:
    print('Raster-to-table key QA/QC passed.')

muaggatt[['mukey', 'muname', 'hydgrpdcd', 'drclassdcd']].merge(
    valu1, on='mukey', how='outer'
)


### Join and Check the AWS Raster

The prepared `aws0_100` raster represents `valu1.aws0_100`, which is already in millimeters. NRCS created this value-added field with strict rules for incomplete horizon data and inconsistent component percentages or depths. The similarly named `muaggatt.aws0100wta` field is in centimeters and can differ because it uses a different aggregation method. We compare the raster with `valu1`, its actual source field. This check can catch a wrong table, key, depth zone, or unit.


In [ ]:
raster_aws_rows = []
for mukey in watershed_mukeys:
    mapunit_mask = (mukey_array == mukey) & valid_aws_mask
    raster_aws_rows.append({
        'mukey': mukey,
        'aws_raster_mean_mm': float(aws_array[mapunit_mask].mean()),
    })

raster_aws_by_mapunit = pd.DataFrame(raster_aws_rows)

mapunit_summary = mapunit_area.merge(
    muaggatt, on='mukey', how='left'
).merge(
    valu1, on='mukey', how='left'
).merge(
    raster_aws_by_mapunit, on='mukey', how='left'
)
mapunit_summary = mapunit_summary.rename(columns={
    'hydgrpdcd': 'dominant_hsg',
    'drclassdcd': 'dominant_drainage_class',
    'aws0_100': 'aws_valu1_0_100cm_mm',
})
mapunit_summary['aws_source_check_difference_mm'] = (
    mapunit_summary['aws_raster_mean_mm']
    - mapunit_summary['aws_valu1_0_100cm_mm']
)

max_aws_difference = mapunit_summary[
    'aws_source_check_difference_mm'
].abs().max()

print(mapunit_summary[[
    'mukey', 'muname', 'dominant_hsg', 'area_acres',
    'aws_raster_mean_mm', 'aws_valu1_0_100cm_mm',
    'aws_source_check_difference_mm',
]].round(2).to_string(index=False))
print(f"\nMaximum raster-table AWS difference: {max_aws_difference:.3f} mm")

if max_aws_difference <= 0.1:
    print('AWS source field, unit, and map-unit join QA/QC passed.')
else:
    print('WARNING: AWS values differ by more than 0.1 mm. Investigate the join and units.')


In [ ]:
point_summary = mapunit_summary[
    mapunit_summary['mukey'] == point_mukey
].copy()
point_summary['point_lon'] = point_lon
point_summary['point_lat'] = point_lat
point_summary['point_aws_0_100cm_mm'] = point_aws_mm

print('Point soil summary:')
print(point_summary[[
    'mukey', 'musym', 'muname', 'dominant_hsg',
    'dominant_drainage_class', 'point_aws_0_100cm_mm',
]].round(2).to_string(index=False))


### Hydrologic Soil Group by Watershed Area

Keep dual groups such as `C/D` as their own category. The first letter represents a drained condition and the second represents the undrained condition. Do not choose the lower-runoff group until effective drainage has been confirmed for the project condition. Missing HSG values also remain visible.


In [ ]:
mapunit_summary['hsg_for_summary'] = (
    mapunit_summary['dominant_hsg'].astype('string').fillna('No data')
)

hsg_area_summary = (
    mapunit_summary
    .groupby('hsg_for_summary', as_index=False)['area_acres']
    .sum()
    .rename(columns={'hsg_for_summary': 'hydrologic_soil_group'})
)
hsg_area_summary['percent_valid_grid'] = (
    100 * hsg_area_summary['area_acres']
    / hsg_area_summary['area_acres'].sum()
)
hsg_area_summary = hsg_area_summary.sort_values(
    'percent_valid_grid', ascending=False
).reset_index(drop=True)

print(hsg_area_summary.round(2).to_string(index=False))


### Watershed Available Water Storage

Every valid cell has equal area in this projected grid, so the simple mean is also an area-weighted mean. We report several statistics because one average does not describe spatial variability.


In [ ]:
aws_valid_values = aws_array[valid_aws_mask].astype(float)

aws_watershed_summary = pd.DataFrame([
    {'statistic': 'Valid grid area', 'value': aws_valid_area_m2 / 4046.86, 'units': 'acres'},
    {'statistic': 'Mean AWS 0-100 cm', 'value': aws_valid_values.mean(), 'units': 'mm'},
    {'statistic': 'Median AWS 0-100 cm', 'value': np.median(aws_valid_values), 'units': 'mm'},
    {'statistic': '10th percentile AWS', 'value': np.percentile(aws_valid_values, 10), 'units': 'mm'},
    {'statistic': '90th percentile AWS', 'value': np.percentile(aws_valid_values, 90), 'units': 'mm'},
])

print(aws_watershed_summary.round(2).to_string(index=False))


### Where Ksat Fits

The Planetary Computer raster collection does not publish Ksat as a direct raster asset. Do not estimate Ksat from AWS or hydrologic soil group. They describe different soil behavior. Lesson 1 remains the course method for depth-weighted and component-weighted Ksat. A full gNATSGO Ksat workflow would require a carefully validated join through component and horizon tables, explicit missing-data coverage, and the same averaging cautions used in Lesson 1.

For H&H modeling, the outputs in this lesson support screening, spatial comparison, and curve-number preparation. They are not calibrated infiltration parameters.


## Part 7: Visualization 📊

Build one categorical map for hydrologic soil group and one continuous map for AWS. The watershed boundary is plotted in the raster CRS so all layers align.


In [ ]:
hsg_colors = {
    'A': '#2c7bb6',
    'B': '#7fcdbb',
    'C': '#fdae61',
    'D': '#d7191c',
    'A/D': '#8c6bb1',
    'B/D': '#88419d',
    'C/D': '#6e016b',
    'No data': '#bdbdbd',
}
hsg_categories = [
    category for category in hsg_colors
    if category in set(mapunit_summary['hsg_for_summary'])
]
hsg_code_lookup = {
    category: index for index, category in enumerate(hsg_categories)
}

hsg_code_array = np.full(mukey_array.shape, np.nan, dtype=float)
for row in mapunit_summary.itertuples():
    hsg_code_array[mukey_array == row.mukey] = hsg_code_lookup[row.hsg_for_summary]

hsg_cmap = ListedColormap([hsg_colors[value] for value in hsg_categories])
aws_masked = np.ma.masked_where(~valid_aws_mask, aws_array)
map_extent = plotting_extent(mukey_array, mukey_transform)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(
    hsg_code_array, cmap=hsg_cmap, extent=map_extent,
    interpolation='nearest',
)
watershed_raster_crs.boundary.plot(ax=axes[0], color='black', linewidth=1.5)
axes[0].set_title('Dominant Hydrologic Soil Group')
axes[0].legend(
    handles=[
        Patch(facecolor=hsg_colors[value], label=value)
        for value in hsg_categories
    ],
    title='HSG', loc='lower left',
)

aws_image = axes[1].imshow(
    aws_masked, cmap='viridis', extent=map_extent,
)
watershed_raster_crs.boundary.plot(ax=axes[1], color='black', linewidth=1.5)
axes[1].set_title('Available Water Storage, 0 to 100 cm')
colorbar = fig.colorbar(aws_image, ax=axes[1], shrink=0.8)
colorbar.set_label('AWS (mm)')

for axis in axes:
    axis.set_xlabel('Easting (m)')
    axis.set_ylabel('Northing (m)')

fig.suptitle(
    f'gNATSGO Soil Screening - HUC-12 {target_huc}',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.savefig('gnatsgo_watershed_soils.png', dpi=200, bbox_inches='tight')
plt.show()


### What This Map Tells Us

The HSG map shows the same broad soil pattern as Lesson 1, but area is now counted on a 10 meter grid. The AWS map shows map-unit weighted values repeated across cells belonging to each map unit. Sharp changes at map-unit boundaries do not represent field-scale measurements. They reflect the mapped soil unit and its aggregated attribute.

The grid makes overlay with land-cover rasters convenient. It does not remove the need to review source scale, dual HSG drainage assumptions, missing values, or calibration evidence.


## Part 8: Export Traceable Outputs 📂

Export the point result, map-unit table, HSG area table, AWS statistics, clipped rasters, and map. Each table records both the source citation and the archived Planetary Computer snapshot date.


In [ ]:
access_date = datetime.date.today().isoformat()
source_citation = (
    'Soil Survey Staff (2026). Gridded national soil survey geographic '
    '(gNATSGO) database. USDA Natural Resources Conservation Service.'
)
distribution = 'Microsoft Planetary Computer gnatsgo-rasters and gnatsgo-tables'

for table in [point_summary, mapunit_summary, hsg_area_summary, aws_watershed_summary]:
    table['huc12'] = target_huc
    table['source_citation'] = source_citation
    table['distribution'] = distribution
    table['dataset_snapshot_date'] = dataset_snapshot_date
    table['access_date'] = access_date

point_summary.to_csv('gnatsgo_point_soil_summary.csv', index=False)
mapunit_summary.to_csv('gnatsgo_mapunit_summary.csv', index=False)
hsg_area_summary.to_csv('gnatsgo_hsg_area_summary.csv', index=False)
aws_watershed_summary.to_csv('gnatsgo_aws_watershed_summary.csv', index=False)

print('Saved four traceable CSV summary files.')


In [ ]:
def save_clipped_raster(filename, array, transform, source_profile, nodata):
    output_profile = source_profile.copy()
    output_profile.update(
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        transform=transform,
        nodata=nodata,
        compress='lzw',
    )
    with rasterio.open(filename, 'w', **output_profile) as dst:
        dst.write(array, 1)


save_clipped_raster(
    'gnatsgo_mukey_clipped.tif', mukey_array, mukey_transform,
    mukey_profile, mukey_nodata,
)
save_clipped_raster(
    'gnatsgo_aws_0_100cm_clipped.tif', aws_array, aws_transform,
    aws_profile, aws_nodata,
)

print('Saved two clipped GeoTIFFs and gnatsgo_watershed_soils.png.')


In [ ]:
# Download all lesson outputs to your computer
from google.colab import files

output_files = [
    'gnatsgo_point_soil_summary.csv',
    'gnatsgo_mapunit_summary.csv',
    'gnatsgo_hsg_area_summary.csv',
    'gnatsgo_aws_watershed_summary.csv',
    'gnatsgo_mukey_clipped.tif',
    'gnatsgo_aws_0_100cm_clipped.tif',
    'gnatsgo_watershed_soils.png',
]

for filename in output_files:
    files.download(filename)


## Engineering Cautions ⚠️

1. **Check the data vintage.** The Planetary Computer collection used here is a July 2020 snapshot. NRCS refreshes gNATSGO annually. Use the latest appropriate release for project work.
2. **A 10 meter cell is not 10 meter soil-survey accuracy.** Grid resolution describes storage. Source survey scale and source type control interpretation.
3. **gNATSGO is a composite.** It uses SSURGO where available and fills gaps with STATSGO2 or RSS data. Detail and confidence can vary across space.
4. **Do not average categorical keys or HSG codes.** Count map-unit or HSG cells by area.
5. **AWS is not Ksat or direct runoff storage.** AWS measures plant-available water over a depth zone. It does not measure saturated flow, total pore space, initial abstraction, or a calibrated loss parameter.
6. **Dual HSG needs a drainage decision.** Confirm whether effective drainage exists before selecting the first letter.
7. **Screening is not calibration.** Soil survey products support initial parameterization and comparison. They do not replace site investigation, infiltration testing, observed flow data, or model calibration.
8. **Do not save signed asset URLs.** Their access tokens expire and should not be committed or shared. Save stable STAC item IDs and source metadata instead.


## Troubleshooting

| Problem | Likely Cause | What to Try |
|---|---|---|
| Catalog or cloud raster request fails | Network issue or temporary service interruption | Check the connection and rerun the catalog, search, and clip cells |
| Signed raster link is rejected | The temporary token expired | Rerun the catalog and STAC search cells to obtain a fresh signed item |
| More than one raster item is found | Watershed crosses a tile boundary | Use a course watershed inside one tile or extend the workflow with `rasterio.merge` |
| Table read is slow | A national Parquet table is being filtered remotely | Wait for the first read and request only needed columns and map-unit keys |
| Raster and `valu1` AWS do not match | Wrong field, key, depth zone, or unit | Compare `aws0_100` with `valu1.aws0_100` in millimeters |
| `muaggatt` and `valu1` AWS differ | The fields use different aggregation rules and units | Do not use `muaggatt.aws0100wta` to validate the prepared `aws0_100` raster |
| Coverage is below 98 percent | NoData, wrong CRS, wrong tile, or clipping problem | Print CRS, bounds, NoData, and valid cell counts before continuing |
| HSG map has grey areas | Some map units have no standard HSG | Keep them as `No data` and review water or miscellaneous map units separately |
| Colab upload fails | Watershed ZIP was not selected | Rerun the upload cell and choose the course ZIP file |


## Practice Exercises 🎯

### Exercise 1: Use Another Course Watershed
Change `target_huc`, rerun the search and clipping workflow, and compare HSG area percentages. Confirm the watershed still intersects one tile.

### Exercise 2: Change the Depth Zone
Replace `aws0_100` with `aws0_30` or `aws0_150`. Update variable names, units, titles, and exports so the depth interval remains clear.

### Exercise 3: Quantify Missing HSG
Calculate the acres and percentage assigned to `No data`. List the map-unit names responsible and decide whether they represent water or another miscellaneous area.

### Exercise 4: Compare Lessons 1 and 2
Compare the HSG area percentages from SSURGO polygons and the gNATSGO grid. Explain small differences caused by grid cells and identify any larger difference that could reflect source or vintage.

### Challenge Exercise: Build a Reusable Function
Create `summarize_gnatsgo_watershed(huc12_code, asset_name)` that returns coverage, area, and continuous-raster statistics. Keep categorical and continuous assets on separate calculation paths.

🤖 **Try asking your AI assistant:** *"Help me design a function that clips one Planetary Computer gNATSGO asset to a watershed. First ask whether the asset is categorical or continuous. Include CRS, NoData, coverage, and unit checks. Do not write code until those assumptions are explicit."*


In [ ]:
# EXERCISE STARTER
# Change this to another HUC-12 from the uploaded course file.
exercise_huc = '101900090106'

# Change this to another continuous gNATSGO asset.
exercise_asset = 'aws0_30'

print(f"Next watershed: {exercise_huc}")
print(f"Next asset: {exercise_asset}")


## Key Takeaways

- gNATSGO provides seamless gridded soil map units assembled from the best available NRCS source data.
- A `mukey` cell is an index that must be joined to soil tables. It is not a quantity to average.
- STAC provides stable discovery while signed URLs provide temporary data access.
- Raster area comes from the affine transform and valid cell count.
- HSG should be summarized by area and dual groups should remain explicit until drainage is confirmed.
- AWS values need depth and units in every label. AWS is not Ksat or a calibrated model parameter.
- Coverage, grid alignment, source date, and raster-to-source-field checks belong in the workflow before interpretation.

## Next Steps

You are ready to combine the HSG grid with Module 8 land cover for a raster-based curve-number workflow. Before design use, confirm the latest NRCS release, inspect source-data lineage, and compare initial parameters with site and calibration evidence.

## Additional Resources

- [NRCS gNATSGO](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo)
- [NRCS SSURGO and STATSGO2 metadata](https://sdmdataaccess.nrcs.usda.gov/documents/TablesAndColumnsReport.pdf)
- [NRCS value-added `valu1` field descriptions](https://www.nrcs.usda.gov/sites/default/files/2022-08/gSSURGO%20Value%20Added%20Look%20Up%20Valu1%20Table%20Column%20Descriptions.pdf)
- [Planetary Computer gNATSGO rasters](https://planetarycomputer.microsoft.com/dataset/gnatsgo-rasters)
- [Planetary Computer tabular data guide](https://planetarycomputer.microsoft.com/docs/quickstarts/reading-tabular-data/)

## That's Lesson 2 Done!

You used a cloud catalog, cloud-optimized rasters, and a Parquet soil table to build a traceable watershed soil summary. More importantly, you kept categorical keys, continuous attributes, units, source vintage, and engineering interpretation separate.
